In [ ]:
#   数据集3：PubMed 数据集的实验
# 采样器：MLP投影到采样空间+score双映射 (效果更合适的版本)

import os
import os.path as osp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv, MessagePassing
from torch_geometric.utils import negative_sampling

# ==========================================
# 模块一：子图增强聚合器 ( GAT )
# ==========================================
class SubgraphEnhancer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index, weights):
        x = self.lin(x)
        return self.propagate(edge_index, x=x, weights=weights)

    def message(self, x_j, weights):
        # weights 由 Gumbel-Softmax 生成，包含梯度信号
        return weights.view(-1, 1) * x_j

# ==========================================
# 模块二：参数化神经采样器 (Bilinear 方案)
# 创新点：特征空间解耦 + 关系交互建模
# ==========================================
class BilinearSampler(nn.Module):
    def __init__(self, in_dim, proj_dim=32):
        super().__init__()
        # 1. 特征投影层：实现采样决策与分类特征的解耦
        self.proj = nn.Linear(in_dim, proj_dim)
        # 2. 交互矩阵 W：学习节点间复杂的非对称或互补关系
        self.W = nn.Parameter(torch.Tensor(proj_dim, proj_dim))
        nn.init.xavier_uniform_(self.W)

    def forward(self, h_i, h_j):
        # 将节点嵌入投影到独立的采样特征空间
        z_i = self.proj(h_i)
        z_j = self.proj(h_j)
        # 执行双线性映射打分: Score = z_i^T * W * z_j
        score = torch.sum((z_i @ self.W) * z_j, dim=-1)
        return torch.sigmoid(score)

# ==========================================
# 模块三：递归多跳协同优化系统 (hops=2)
# ==========================================
class NeuralRecursiveSystem(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size, hops=2, tau=0.8): # 需要考虑退火策略
        super().__init__()
        # 1. 基础特征提取
        self.gat = GATConv(in_size, hidden_size, heads=8, dropout=0.6)
        
        # 2. 双线性映射采样器
        self.sampler_net = BilinearSampler(in_dim=hidden_size * 8)
        
        # 3. 增强器与预测任务
        self.enhancer = SubgraphEnhancer(hidden_size * 8, hidden_size * 8)
        self.link_predictor = torch.nn.Sequential(
            torch.nn.Linear((hidden_size * 8) * 2, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
            torch.nn.Sigmoid()
        )
        self.classifier = torch.nn.Linear(hidden_size * 8, out_size)
        
        self.hops = hops
        self.tau = tau

    def get_neural_recursive_weights(self, h, edge_index, start_mask):
        row, col = edge_index
        num_nodes = h.size(0)
        num_edges = edge_index.size(1)
        
        # 计算全局潜在边得分
        scores = self.sampler_net(h[row], h[col])
        
        final_weights = torch.zeros(num_edges, device=edge_index.device)
        active_nodes = start_mask.clone() # 起始种子节点
        
        for h_step in range(self.hops):
            active_edges_mask = active_nodes[row]
            if not active_edges_mask.any():
                break
            
            # 构造 Gumbel-Softmax 分布
            current_scores = scores * active_edges_mask.float()
            sampling_logits = torch.stack([1 - current_scores, current_scores], dim=-1).clamp(min=1e-9).log()
            
            # 执行硬采样同时保持梯度回传
            sampling_mask = F.gumbel_softmax(sampling_logits, tau=self.tau, hard=True, dim=-1)[:, 1]
            
            # 更新全局子图权重
            final_weights = torch.max(final_weights, sampling_mask * active_edges_mask.float())
            
            # 【生长逻辑】：使用并集(|)确保父节点及其选中的邻居在下一跳都活跃，防止断流
            new_active_nodes = torch.zeros(num_nodes, device=edge_index.device, dtype=torch.bool)
            new_active_nodes[col[sampling_mask > 0.5]] = True
            active_nodes = active_nodes | new_active_nodes 

        return final_weights

    def forward(self, x, edge_index, train_mask):
        # 1. 获取基础表征
        h_base = F.elu(self.gat(x, edge_index))
        
        # 2. 神经递归采样 (Bilinear + Gumbel)
        weights = self.get_neural_recursive_weights(h_base, edge_index, train_mask)
        
        # 3. 特征增强 (引入残差连接以增强训练稳定性)
        h_enhanced = h_base
        for _ in range(self.hops):
            # 增强特征 = 上一轮特征 + 采样的邻居聚合信息
            h_enhanced = h_enhanced + F.elu(self.enhancer(h_enhanced, edge_index, weights))
        
        # 4. 链路预测 (正样本)
        row, col = edge_index
        pos_edge_feat = torch.cat([h_enhanced[row], h_enhanced[col]], dim=-1)
        link_probs_pos = self.link_predictor(pos_edge_feat).squeeze()
        
        # 5. 节点分类
        logits = self.classifier(h_enhanced)
        
        return F.log_softmax(logits, dim=1), link_probs_pos, h_enhanced, weights

# ==========================================
# 模块四：训练循环与评估 (引入tau退火)
# ==========================================

#  Cora 数据集
#path = osp.join(os.getcwd(), 'data', 'Cora')
#dataset = Planetoid(root=path, name='Cora')
#data = dataset[0]

# citeceer 数据集

#path = osp.join(os.getcwd(), 'data', 'CiteSeer')
#dataset = Planetoid(root=path, name='CiteSeer')
#data = dataset[0]

# PubMed数据集
path = osp.join(os.getcwd(), 'data', 'PubMed')
dataset = Planetoid(root=path, name='PubMed')
data = dataset[0]

# 初始化模型
model = NeuralRecursiveSystem(dataset.num_features, 8, dataset.num_classes, hops=2) # 修改跳数
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        # 获取模型输出
        log_probs, _, _, _ = model(data.x, data.edge_index, data.train_mask)
        
        # 得到预测结果：概率最大的索引
        pred = log_probs[mask].max(1)[1]
        
        # 计算正确数量
        correct = pred.eq(data.y[mask]).sum().item()
        
        # 计算准确率
        acc = correct / mask.sum().item()
        return acc

# 初始化 history 字典用于存储数据
history = {'loss': [], 'clf_loss': [], 'link_loss': []}

def train(current_tau): # 增加 tau 参数
    model.train()
    model.tau = current_tau # 在每一轮训练前更新模型的温度
    optimizer.zero_grad()
    
    # 前向传播
    log_probs, link_probs_pos, h_enhanced, weights = model(data.x, data.edge_index, data.train_mask)
    
    # 负采样用于链路预测训练
    neg_edge_index = negative_sampling(data.edge_index, num_nodes=data.num_nodes)
    neg_row, neg_col = neg_edge_index
    neg_edge_feat = torch.cat([h_enhanced[neg_row], h_enhanced[neg_col]], dim=-1)
    link_probs_neg = model.link_predictor(neg_edge_feat).squeeze()
    
    # 计算复合损失
    loss_clf = F.nll_loss(log_probs[data.train_mask], data.y[data.train_mask])
    loss_link = F.binary_cross_entropy(link_probs_pos, torch.ones_like(link_probs_pos)) + \
                F.binary_cross_entropy(link_probs_neg, torch.zeros_like(link_probs_neg))
    
    loss_sparse = weights.mean() 
    # 针对 CiteSeer 的建议：如果准确率低，可以尝试把 loss_sparse 权重设为 0.001 或者暂时不加
    total_loss = loss_clf + 0.8 * loss_link 
    
    total_loss.backward()
    optimizer.step()
    
    return total_loss.item(), loss_clf.item(), loss_link.item()

# 运行训练并记录数据
print("开始训练（使用 Tau 温度退火）...")

# 退火参数设置
start_tau = 1  # 初始高温度，允许采样器广泛探索
end_tau = 0.1    # 最终低温度，使采样趋于确定（硬化）
epochs = 150

for epoch in range(1, epochs + 1):
    # 计算当前轮次的温度：线性退火
    curr_tau = max(end_tau, start_tau - (epoch / epochs) * (start_tau - end_tau))
    
    t_loss, c_loss, l_loss = train(curr_tau) # 传入当前温度
    
    history['loss'].append(t_loss)
    history['clf_loss'].append(c_loss)
    history['link_loss'].append(l_loss)
    
    if epoch % 10 == 0:
        # 在打印时同时查看测试集准确率，观察退火效果
        test_acc_epoch = evaluate(data.test_mask)
        print(f"Epoch: {epoch:03d} | Tau: {curr_tau:.2f} | Total Loss: {t_loss:.4f} | Test Acc: {test_acc_epoch:.4f}")


# 最终报告
model.eval()
with torch.no_grad():
    _, _, _, final_weights = model(data.x, data.edge_index, data.train_mask)
    total_active_edges = (final_weights > 0.5).sum().item()
    print(f"\n训练完成。子图采样器最终激活了 {total_active_edges} 条边。")


开始训练（使用 Tau 温度退火）...
Epoch: 010 | Tau: 0.94 | Total Loss: 1.9236 | Test Acc: 0.4290
Epoch: 020 | Tau: 0.88 | Total Loss: 1.3291 | Test Acc: 0.5510
Epoch: 030 | Tau: 0.82 | Total Loss: 1.0049 | Test Acc: 0.6050
Epoch: 040 | Tau: 0.76 | Total Loss: 0.9803 | Test Acc: 0.6710
Epoch: 050 | Tau: 0.70 | Total Loss: 0.9932 | Test Acc: 0.7040
Epoch: 060 | Tau: 0.64 | Total Loss: 0.9474 | Test Acc: 0.6590
Epoch: 070 | Tau: 0.58 | Total Loss: 0.9259 | Test Acc: 0.7060
Epoch: 080 | Tau: 0.52 | Total Loss: 0.9087 | Test Acc: 0.7190
Epoch: 090 | Tau: 0.46 | Total Loss: 0.8839 | Test Acc: 0.7200
Epoch: 100 | Tau: 0.40 | Total Loss: 0.8559 | Test Acc: 0.7170
Epoch: 110 | Tau: 0.34 | Total Loss: 0.8220 | Test Acc: 0.7080
Epoch: 120 | Tau: 0.28 | Total Loss: 0.7964 | Test Acc: 0.6880
Epoch: 130 | Tau: 0.22 | Total Loss: 0.7573 | Test Acc: 0.6900
Epoch: 140 | Tau: 0.16 | Total Loss: 0.7136 | Test Acc: 0.6890
Epoch: 150 | Tau: 0.10 | Total Loss: 0.6699 | Test Acc: 0.6890

训练完成。子图采样器最终激活了 4126 条边。


In [32]:
#  PubMed 数据集的实验
# 采样器：MLP投影到采样空间+score双映射 (效果更合适的版本)
# LOSS 加入稀疏性惩罚

import os
import os.path as osp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GATConv, MessagePassing
from torch_geometric.utils import negative_sampling

# ==========================================
# 模块一：子图增强聚合器 ( GAT )
# ==========================================
class SubgraphEnhancer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = torch.nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index, weights):
        x = self.lin(x)
        return self.propagate(edge_index, x=x, weights=weights)

    def message(self, x_j, weights):
        # weights 由 Gumbel-Softmax 生成，包含梯度信号
        return weights.view(-1, 1) * x_j

# ==========================================
# 模块二：参数化神经采样器 (Bilinear 方案)
# 创新点：特征空间解耦 + 关系交互建模
# ==========================================
class BilinearSampler(nn.Module):
    def __init__(self, in_dim, proj_dim=32):
        super().__init__()
        # 1. 特征投影层：实现采样决策与分类特征的解耦
        self.proj = nn.Linear(in_dim, proj_dim)
        # 2. 交互矩阵 W：学习节点间复杂的非对称或互补关系
        self.W = nn.Parameter(torch.Tensor(proj_dim, proj_dim))
        nn.init.xavier_uniform_(self.W)

    def forward(self, h_i, h_j):
        # 将节点嵌入投影到独立的采样特征空间
        z_i = self.proj(h_i)
        z_j = self.proj(h_j)
        # 执行双线性映射打分: Score = z_i^T * W * z_j
        score = torch.sum((z_i @ self.W) * z_j, dim=-1)
        return torch.sigmoid(score)

# ==========================================
# 模块三：递归多跳协同优化系统 (hops=2)
# ==========================================
class NeuralRecursiveSystem(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size, hops=2, tau=0.8): 
        super().__init__()
        # 1. 基础特征提取 (第一阶段：语义发掘)
        self.gat = GATConv(in_size, hidden_size, heads=8, dropout=0.6)
        
        # 2. 双线性映射采样器 (第二阶段：采样决策)
        self.sampler_net = BilinearSampler(in_dim=hidden_size * 8)
        
        # 3. 增强器与预测任务 (第三阶段：任务验证)
        self.enhancer = SubgraphEnhancer(hidden_size * 8, hidden_size * 8)
        self.link_predictor = torch.nn.Sequential(
            torch.nn.Linear((hidden_size * 8) * 2, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
            torch.nn.Sigmoid()
        )
        self.classifier = torch.nn.Linear(hidden_size * 8, out_size)
        
        self.hops = hops
        self.tau = tau

    def get_neural_recursive_weights(self, h, edge_index, start_mask):
        row, col = edge_index
        num_nodes = h.size(0)
        num_edges = edge_index.size(1)
        
        # 计算全局潜在边得分
        scores = self.sampler_net(h[row], h[col])
        
        final_weights = torch.zeros(num_edges, device=edge_index.device)
        active_nodes = start_mask.clone() # 起始种子节点
        
        for h_step in range(self.hops):
            active_edges_mask = active_nodes[row]
            if not active_edges_mask.any():
                break
            
            # 构造 Gumbel-Softmax 分布
            current_scores = scores * active_edges_mask.float()
            sampling_logits = torch.stack([1 - current_scores, current_scores], dim=-1).clamp(min=1e-9).log()
            
            # 执行硬采样同时保持梯度回传
            sampling_mask = F.gumbel_softmax(sampling_logits, tau=self.tau, hard=True, dim=-1)[:, 1]
            
            # 更新全局子图权重
            final_weights = torch.max(final_weights, sampling_mask * active_edges_mask.float())
            
            # 【生长逻辑】：使用并集(|)确保父节点及其选中的邻居在下一跳都活跃
            new_active_nodes = torch.zeros(num_nodes, device=edge_index.device, dtype=torch.bool)
            new_active_nodes[col[sampling_mask > 0.5]] = True
            active_nodes = active_nodes | new_active_nodes 

        return final_weights

    def forward(self, x, edge_index, train_mask):
        # 1. 获取基础表征 (语义发掘)
        h_base = F.elu(self.gat(x, edge_index))
        
        # 2. 神经递归采样 (采样决策)
        weights = self.get_neural_recursive_weights(h_base, edge_index, train_mask)
        
        # 3. 特征增强 (任务验证准备)
        h_enhanced = h_base
        for _ in range(self.hops):
            h_enhanced = h_enhanced + F.elu(self.enhancer(h_enhanced, edge_index, weights))
        
        # 4. 链路预测 (任务验证)
        row, col = edge_index
        pos_edge_feat = torch.cat([h_enhanced[row], h_enhanced[col]], dim=-1)
        link_probs_pos = self.link_predictor(pos_edge_feat).squeeze()
        
        # 5. 节点分类 (语义锚点)
        logits = self.classifier(h_enhanced)
        
        return F.log_softmax(logits, dim=1), link_probs_pos, h_enhanced, weights

# ==========================================
# 模块四：训练循环与评估
# ==========================================

# PubMed数据集
path = osp.join(os.getcwd(), 'data', 'PubMed')
dataset = Planetoid(root=path, name='PubMed')
data = dataset[0]

# 初始化模型
model = NeuralRecursiveSystem(dataset.num_features, 8, dataset.num_classes, hops=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=5e-4)

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        log_probs, _, _, _ = model(data.x, data.edge_index, data.train_mask)
        pred = log_probs[mask].max(1)[1]
        correct = pred.eq(data.y[mask]).sum().item()
        acc = correct / mask.sum().item()
        return acc

# 初始化记录
history = {'loss': [], 'clf_loss': [], 'link_loss': [], 'sparse_loss': []}

# --- 训练核心逻辑：引入显式稀疏度约束 ---
def train(current_tau):
    model.train()
    model.tau = current_tau
    optimizer.zero_grad()
    
    # 1. 前向传播：语义 -> 采样 -> 任务
    log_probs, link_probs_pos, h_enhanced, weights = model(data.x, data.edge_index, data.train_mask)
    
    # 2. 负采样用于链路预测辅助任务
    neg_edge_index = negative_sampling(data.edge_index, num_nodes=data.num_nodes)
    neg_row, neg_col = neg_edge_index
    neg_edge_feat = torch.cat([h_enhanced[neg_row], h_enhanced[neg_col]], dim=-1)
    link_probs_neg = model.link_predictor(neg_edge_feat).squeeze()
    
    # 3. Loss 组成规划
    # A. 语义损失 (锚点) # 防止语义漂移（对采样 enhanced 的 embedding）
    loss_clf = F.nll_loss(log_probs[data.train_mask], data.y[data.train_mask])
    
    # B. 链路预测损失 (下游反馈：修正采样器的关键梯度来源)
    loss_link = F.binary_cross_entropy(link_probs_pos, torch.ones_like(link_probs_pos)) + \
                F.binary_cross_entropy(link_probs_neg, torch.zeros_like(link_probs_neg))
    
    # C. 显式稀疏度损失 (采样阶段独立约束)
    # 创新逻辑：使用目标稀疏度正则化，迫使模型在“有限预算”下寻找最优边
    target_sparsity = 0.15  # 目标保留 15% 的边
    loss_sparse = torch.abs(weights.mean() - target_sparsity)
    
    # 4. 全局 Loss 协同优化
    # 这里的权重分配 (0.8, 0.1) 旨在保持下游任务占据主导，同时施加结构压力
    total_loss = loss_clf + 0.8 * loss_link + 5 * loss_sparse
    
    total_loss.backward()
    optimizer.step()
    
    return total_loss.item(), loss_clf.item(), loss_link.item(), loss_sparse.item()

# 运行训练
print("开始训练（Tau退火 + 显式稀疏约束）...")

start_tau = 1
end_tau = 0.1    
epochs = 150

for epoch in range(1, epochs + 1):
    curr_tau = max(end_tau, start_tau - (epoch / epochs) * (start_tau - end_tau))
    
    t_loss, c_loss, l_loss, s_loss = train(curr_tau)
    
    history['loss'].append(t_loss)
    history['clf_loss'].append(c_loss)
    history['link_loss'].append(l_loss)
    history['sparse_loss'].append(s_loss)
    
    if epoch % 10 == 0:
        test_acc_epoch = evaluate(data.test_mask)
        print(f"Epoch: {epoch:03d} | Tau: {curr_tau:.2f} | Total Loss: {t_loss:.4f} | Sparse Loss: {s_loss:.4f} | Test Acc: {test_acc_epoch:.4f}")

# 最终报告
model.eval()
with torch.no_grad():
    _, _, _, final_weights = model(data.x, data.edge_index, data.train_mask)
    total_active_edges = (final_weights > 0.5).sum().item()
    sparsity = total_active_edges / data.edge_index.size(1)
    print(f"\n训练完成。")
    print(f"最终激活边数: {total_active_edges} / {data.edge_index.size(1)}")
    print(f"最终子图稀疏度: {sparsity:.4%}")

开始训练（Tau退火 + 显式稀疏约束）...
Epoch: 010 | Tau: 0.94 | Total Loss: 2.5027 | Sparse Loss: 0.1313 | Test Acc: 0.5960
Epoch: 020 | Tau: 0.88 | Total Loss: 1.6469 | Sparse Loss: 0.1035 | Test Acc: 0.6460
Epoch: 030 | Tau: 0.82 | Total Loss: 1.7758 | Sparse Loss: 0.1034 | Test Acc: 0.6890
Epoch: 040 | Tau: 0.76 | Total Loss: 1.5147 | Sparse Loss: 0.1034 | Test Acc: 0.6990
Epoch: 050 | Tau: 0.70 | Total Loss: 1.4967 | Sparse Loss: 0.1034 | Test Acc: 0.7130
Epoch: 060 | Tau: 0.64 | Total Loss: 1.4830 | Sparse Loss: 0.1034 | Test Acc: 0.7210
Epoch: 070 | Tau: 0.58 | Total Loss: 1.4608 | Sparse Loss: 0.1034 | Test Acc: 0.7200
Epoch: 080 | Tau: 0.52 | Total Loss: 1.4409 | Sparse Loss: 0.1034 | Test Acc: 0.7290
Epoch: 090 | Tau: 0.46 | Total Loss: 1.4201 | Sparse Loss: 0.1034 | Test Acc: 0.7380
Epoch: 100 | Tau: 0.40 | Total Loss: 1.3924 | Sparse Loss: 0.1035 | Test Acc: 0.7410
Epoch: 110 | Tau: 0.34 | Total Loss: 1.3516 | Sparse Loss: 0.1035 | Test Acc: 0.7370
Epoch: 120 | Tau: 0.28 | Total Loss: 1.30

In [ ]:
# 续上
# ==========================================
# 模块五：可视化扩展工具 
# ==========================================
import matplotlib.pyplot as plt
import networkx as nx

def plot_training_results(history):
    """可视化训练 Loss 曲线"""
    plt.figure(figsize=(10, 5))
    # 修复点：除了总 Loss，同时展示分类和链路预测 Loss 更有助于分析
    plt.plot(history['loss'], label='Total Loss', color='tab:orange', linewidth=2)
    plt.plot(history['clf_loss'], label='Classification Loss', color='tab:blue', linestyle='--')
    plt.plot(history['link_loss'], label='Link Prediction Loss', color='tab:green', linestyle=':')
    
    plt.title('Training Loss ', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss Value', fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


# ==========================================
# 模块六：执行可视化 
# ==========================================
# 1. 绘制刚才训练记录的 Loss 曲线
plot_training_results(history)

# 2. 选择一个训练集节点进行 2-hop 采样展示
#available_nodes = data.train_mask.nonzero(as_tuple=True)[0]
#if len(available_nodes) > 0:
    #seed_node = available_nodes[0].item()
    #print(f"正在生成节点 {seed_node} 的 2 跳递归采样可视化图...")
    #visualize_2hop_sampling(model, data, seed_node)

def evaluate(mask):
    model.eval()
    with torch.no_grad():
        # 获取模型输出
        log_probs, _, _, _ = model(data.x, data.edge_index, data.train_mask)
        
        # 得到预测结果：概率最大的索引
        pred = log_probs[mask].max(1)[1]
        
        # 计算正确数量
        correct = pred.eq(data.y[mask]).sum().item()
        
        # 计算准确率
        acc = correct / mask.sum().item()
        return acc

# 在训练循环结束后调用
test_acc = evaluate(data.test_mask)
print(f"最终测试集准确率: {test_acc:.4f}")



In [ ]:
# 可视化采样子图

import networkx as nx
import matplotlib.pyplot as plt
from torch_geometric.utils import k_hop_subgraph

def visualize_recursive_sampling(seed_node, edge_index, final_weights, hops=3):
    """
    可视化特定种子节点的递归采样结果
    红色：种子节点 | 粉色：被采样进来的邻居 | 灰色：物理邻居但未被采样
    """
    # 1. 提取该种子节点的 k-hop 物理子图范围
    subset, sub_edge_index, mapping, edge_mask = k_hop_subgraph(
        seed_node, hops, edge_index, relabel_nodes=True
    )
    
    # 2. 获取该子图范围内的采样权重
    sub_weights = final_weights[edge_mask]
    
    # 3. 构建 NetworkX 图对象
    G = nx.Graph()
    node_list = subset.tolist()
    G.add_nodes_from(range(len(node_list)))
    
    # 添加边并标记属性
    for i in range(sub_edge_index.size(1)):
        u, v = sub_edge_index[0, i].item(), sub_edge_index[1, i].item()
        is_sampled = sub_weights[i] > 0.99
        G.add_edge(u, v, sampled=is_sampled)

    # 4. 确定节点颜色逻辑
    # 种子节点 (Center) -> 红色
    # 采样可达节点 -> 粉色
    # 物理邻居但采样不可达 -> 灰色
    center_idx = mapping.item()
    node_colors = []
    
    # 使用简单的可达性分析确定哪些点是通过采样边连进来的
    sampled_subgraph_nodes = {center_idx}
    for _ in range(hops):
        for u, v, d in G.edges(data=True):
            if d['sampled']:
                if u in sampled_subgraph_nodes: sampled_subgraph_nodes.add(v)
                if v in sampled_subgraph_nodes: sampled_subgraph_nodes.add(u)

    for i in range(len(node_list)):
        if i == center_idx:
            node_colors.append('red')
        elif i in sampled_subgraph_nodes:
            node_colors.append('pink')
        else:
            node_colors.append('lightgrey')

    # 5. 绘图设置
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(G, k=0.3, seed=42) # 布局算法
    
    # 绘制边：采样边用粉色粗线，未采样边用灰色细虚线
    sampled_edges = [(u, v) for u, v, d in G.edges(data=True) if d['sampled']]
    unused_edges = [(u, v) for u, v, d in G.edges(data=True) if not d['sampled']]
    
    nx.draw_networkx_edges(G, pos, edgelist=sampled_edges, width=2, edge_color='hotpink', alpha=0.8)
    nx.draw_networkx_edges(G, pos, edgelist=unused_edges, width=1, edge_color='lightgrey', style='dashed', alpha=0.4)
    
    # 绘制节点
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=400)
    
    plt.title(f"Recursive Sampling Subgraph (Seed: {seed_node}, Hops: {hops})")
    plt.show()

# ==========================================
# 调用可视化 (在训练结束后运行)
# ==========================================
model.eval()
with torch.no_grad():
    # 随机选择一个训练集中的种子节点进行观察
    available_seeds = data.train_mask.nonzero().view(-1).tolist()
    example_seed = available_seeds[0] # 取训练集第一个节点
    
    _, _, _, final_weights = model(data.x, data.edge_index, data.train_mask)
    
    print(f"正在生成种子节点 {example_seed} 的采样可视化图...")
    visualize_recursive_sampling(example_seed, data.edge_index, final_weights, hops=model.hops)

In [5]:
# 数据集4：Amazon （loss soft lambda）

import os
import os.path as osp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Amazon
from torch_geometric.nn import GATConv, MessagePassing
from torch_geometric.utils import negative_sampling
import numpy as np

# ==========================================
# 模块一：SubgraphEnhancer
# ==========================================
class SubgraphEnhancer(MessagePassing):
    def __init__(self, in_channels, out_channels):
        super().__init__(aggr='add')
        self.lin = torch.nn.Linear(in_channels, out_channels)
    def forward(self, x, edge_index, weights):
        x = self.lin(x)
        return self.propagate(edge_index, x=x, weights=weights)
    def message(self, x_j, weights):
        return weights.view(-1, 1) * x_j

# ==========================================
# 模块二：BilinearSampler (带偏置 防塌方)
# ==========================================
class BilinearSampler(nn.Module):
    def __init__(self, in_dim, proj_dim=32):
        super().__init__()
        self.proj = nn.Linear(in_dim, proj_dim)
        self.W = nn.Parameter(torch.Tensor(proj_dim, proj_dim))
        nn.init.xavier_uniform_(self.W)

    def forward(self, h_i, h_j):
        z_i = self.proj(h_i)
        z_j = self.proj(h_j)
        # 增加 baseline 偏置 +1.0，确保初始概率偏向 1
        score = torch.sum((z_i @ self.W) * z_j, dim=-1) + 1.0 
        return torch.sigmoid(score)

# ==========================================
# 模块三：Neural Recursive System 
# ==========================================
class NeuralRecursiveSystem(torch.nn.Module):
    def __init__(self, in_size, hidden_size, out_size, hops=2, tau=0.8): 
        super().__init__()
        self.gat = GATConv(in_size, hidden_size, heads=8, dropout=0.6)
        self.sampler_net = BilinearSampler(in_dim=hidden_size * 8)
        self.enhancer = SubgraphEnhancer(hidden_size * 8, hidden_size * 8)
        self.link_predictor = torch.nn.Sequential(
            torch.nn.Linear((hidden_size * 8) * 2, 32),
            torch.nn.ReLU(),
            torch.nn.Linear(32, 1),
            torch.nn.Sigmoid()
        )
        # 多层 MLP
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_size * 8, hidden_size * 4),
            torch.nn.BatchNorm1d(hidden_size * 4),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.5),
            torch.nn.Linear(hidden_size * 4, out_size)
        )
        # self.classifier = torch.nn.Linear(hidden_size * 8, out_size)
        self.hops = hops
        self.tau = tau

    def get_neural_recursive_weights(self, h, edge_index, start_mask, hard=True):
        row, col = edge_index
        num_nodes = h.size(0)
        num_edges = edge_index.size(1)
        scores = self.sampler_net(h[row], h[col])
        final_weights = torch.zeros(num_edges, device=edge_index.device)
        active_nodes = start_mask.clone() 
        
        for h_step in range(self.hops):
            active_edges_mask = active_nodes[row]
            if not active_edges_mask.any(): break
            
            current_scores = scores * active_edges_mask.float()
            sampling_logits = torch.stack([1 - current_scores, current_scores], dim=-1).clamp(min=1e-9).log()
            
            # 核心：使用 Gumbel-Softmax
            sampling_mask = F.gumbel_softmax(sampling_logits, tau=self.tau, hard=hard, dim=-1)[:, 1]
            
            final_weights = torch.max(final_weights, sampling_mask * active_edges_mask.float())
            new_active_nodes = torch.zeros(num_nodes, device=edge_index.device, dtype=torch.bool)
            
            # 这里的阈值判断保持 0.5，但在 soft 阶段它依然会传递连续梯度
            new_active_nodes[col[sampling_mask > 0.5]] = True
            active_nodes = active_nodes | new_active_nodes 
        return final_weights

    def forward(self, x, edge_index, train_mask, hard=True):
        h_base = F.elu(self.gat(x, edge_index))
        weights = self.get_neural_recursive_weights(h_base, edge_index, train_mask, hard=hard)
        
        h_enhanced = h_base
        for _ in range(self.hops):
            h_enhanced = h_enhanced + F.elu(self.enhancer(h_enhanced, edge_index, weights))
        
        row, col = edge_index
        pos_edge_feat = torch.cat([h_enhanced[row], h_enhanced[col]], dim=-1)
        link_probs_pos = self.link_predictor(pos_edge_feat).squeeze()
        logits = self.classifier(h_enhanced)
        return F.log_softmax(logits, dim=1), link_probs_pos, h_enhanced, weights

# ==========================================
# 模块四：数据加载 
# ==========================================
dataset_name = 'Computers'
path = osp.join(os.getcwd(), 'data', dataset_name)
dataset = Amazon(root=path, name=dataset_name)
data = dataset[0]

num_nodes = data.num_nodes
indices = torch.randperm(num_nodes)
data.train_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.test_mask = torch.zeros(num_nodes, dtype=torch.bool)
data.train_mask[indices[:int(num_nodes * 0.6)]] = True
data.test_mask[indices[int(num_nodes * 0.8):]] = True

model = NeuralRecursiveSystem(dataset.num_features, 8, dataset.num_classes, hops=2)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)

# ==========================================
# 模块五：训练逻辑
# ==========================================

def train(epoch, total_epochs, start_tau, end_tau):
    model.train()
    
    # 1. 采样模式调度
    use_hard = True if epoch > 50 else False
    
    # 2. Tau 线性退火
    curr_tau = max(end_tau, start_tau - (epoch / total_epochs) * (start_tau - end_tau))
    model.tau = curr_tau

    # 3. Lambda 动态调度
    if epoch <= 20:
        curr_lambda = 0.1
    elif epoch <= 50:
        progress = (epoch - 20) / 30 # 修正分母以匹配 50 轮切换
        curr_lambda = 0.1 + progress * (5.0 - 0.1)
    else:
        curr_lambda = 5.0

    optimizer.zero_grad()
    log_probs, link_probs_pos, h_enhanced, weights = model(data.x, data.edge_index, data.train_mask, hard=use_hard)
    
    # Loss 计算
    neg_edge_index = negative_sampling(data.edge_index, num_nodes=data.num_nodes)
    neg_row, neg_col = neg_edge_index
    neg_edge_feat = torch.cat([h_enhanced[neg_row], h_enhanced[neg_col]], dim=-1)
    link_probs_neg = model.link_predictor(neg_edge_feat).squeeze()
    
    loss_clf = F.nll_loss(log_probs[data.train_mask], data.y[data.train_mask])
    loss_link = F.binary_cross_entropy(link_probs_pos, torch.ones_like(link_probs_pos)) + \
                F.binary_cross_entropy(link_probs_neg, torch.zeros_like(link_probs_neg))
    
    current_sp = weights.mean()
    target_sparsity = 0.30
    loss_sparse = F.mse_loss(current_sp, torch.tensor(target_sparsity).to(weights.device))
    
    total_loss = loss_clf + 1.0 * loss_link + curr_lambda * loss_sparse
    if current_sp < 0.05:
        total_loss += (0.1 - current_sp) * 20 

    total_loss.backward()

    # ====== 梯度监控 ======
    grad_norm = 0.0 # 在这里初始化，确保函数一定会返回它
    with torch.no_grad():
        for param in model.sampler_net.parameters():
            if param.grad is not None:
                grad_norm += param.grad.norm(2).item()
    # ===============================

    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    actual_sparsity = (weights > 0.5).float().mean().item()
    # 增加 grad_norm 到返回值中
    return total_loss.item(), loss_clf.item(), loss_link.item(), loss_sparse.item(), actual_sparsity, curr_lambda, grad_norm

# 运行实验
print(f"开始 Amazon {dataset_name} 训练...")
epochs = 400
for epoch in range(1, epochs + 1):
    # 增加 g_norm 来接收返回的第 7 个值
    t_loss, c_loss, l_loss, s_loss, sp_rate, c_lam, g_norm = train(epoch, epochs, 1.0, 0.1)
    
    mode_str = 'Hard' if epoch > 50 else 'Soft'
    
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            lp, _, _, _ = model(data.x, data.edge_index, data.train_mask, hard=True)
            acc = (lp[data.test_mask].max(1)[1] == data.y[data.test_mask]).float().mean().item()
        
        print(f"Ep: {epoch:03d} | Mode: {mode_str} | λ: {c_lam:.2f} | Sparsity: {sp_rate:.4f} | Test Acc: {acc:.4f}")
    
    # g_norm
    #if epoch % 5 == 0:
        #print(f"Epoch {epoch:03d} | Mode: {mode_str} | Sampler Grad Norm: {g_norm:.6f}")

开始 Amazon Computers 训练...
Ep: 010 | Mode: Soft | λ: 0.10 | Sparsity: 0.2656 | Test Acc: 0.0811
Ep: 020 | Mode: Soft | λ: 0.10 | Sparsity: 0.0637 | Test Acc: 0.0574
Ep: 030 | Mode: Soft | λ: 1.73 | Sparsity: 0.0669 | Test Acc: 0.0654
Ep: 040 | Mode: Soft | λ: 3.37 | Sparsity: 0.0700 | Test Acc: 0.3108
Ep: 050 | Mode: Soft | λ: 5.00 | Sparsity: 0.0829 | Test Acc: 0.3501
Ep: 060 | Mode: Hard | λ: 5.00 | Sparsity: 0.3330 | Test Acc: 0.3748
Ep: 070 | Mode: Hard | λ: 5.00 | Sparsity: 0.2346 | Test Acc: 0.3148
Ep: 080 | Mode: Hard | λ: 5.00 | Sparsity: 0.2819 | Test Acc: 0.4340
Ep: 090 | Mode: Hard | λ: 5.00 | Sparsity: 0.3173 | Test Acc: 0.4311
Ep: 100 | Mode: Hard | λ: 5.00 | Sparsity: 0.3294 | Test Acc: 0.4547
Ep: 110 | Mode: Hard | λ: 5.00 | Sparsity: 0.4232 | Test Acc: 0.4773
Ep: 120 | Mode: Hard | λ: 5.00 | Sparsity: 0.3381 | Test Acc: 0.5249
Ep: 130 | Mode: Hard | λ: 5.00 | Sparsity: 0.3336 | Test Acc: 0.5387
Ep: 140 | Mode: Hard | λ: 5.00 | Sparsity: 0.3526 | Test Acc: 0.5580
Ep: 150 

In [7]:
# 评估采样密度、全局link和采样器link

@torch.no_grad()
def evaluate_density_and_link_impact():
    model.eval()
    
    # 1. 前向传播：获取增强后的表征 h_enhanced 和采样权重 weights
    # 这里的 h_enhanced 是经过了递归采样和 SubgraphEnhancer 聚合后的特征
    log_probs, link_probs_pos, h_enhanced, weights = model(data.x, data.edge_index, data.train_mask)
    
    # 2. 计算密度 (Density)
    total_edges = data.edge_index.size(1)
    active_edges = (weights > 0.5).sum().item()
    density = active_edges / total_edges
    
    # 3. 构造链路预测的测试集 (包含正样本边和负样本边)
    # 正样本：原图中的边
    # 负样本：随机采样的不存在的边
    neg_edge_index = negative_sampling(data.edge_index, num_nodes=data.num_nodes)
    neg_row, neg_col = neg_edge_index
    
    # --- 方案 A：基于采样增强特征的 Link Prediction ---
    # 使用经过采样器筛选、增强后的 h_enhanced 进行预测
    pos_enhanced_feat = torch.cat([h_enhanced[data.edge_index[0]], h_enhanced[data.edge_index[1]]], dim=-1)
    neg_enhanced_feat = torch.cat([h_enhanced[neg_row], h_enhanced[neg_col]], dim=-1)
    
    lp_pos_s = model.link_predictor(pos_enhanced_feat).squeeze()
    lp_neg_s = model.link_predictor(neg_enhanced_feat).squeeze()
    
    y_true = torch.cat([torch.ones(lp_pos_s.size(0)), torch.zeros(lp_neg_s.size(0))])
    y_pred_s = torch.cat([lp_pos_s, lp_neg_s])
    sampler_impact_acc = ((y_pred_s > 0.5).float() == y_true).float().mean().item()
    
    # --- 方案 B：基于原始全局特征的 Link Prediction (基准) ---
    # 使用未经采样增强的原始 GAT 输出 h_base 进行预测
    h_base = F.elu(model.gat(data.x, data.edge_index))
    pos_base_feat = torch.cat([h_base[data.edge_index[0]], h_base[data.edge_index[1]]], dim=-1)
    neg_base_feat = torch.cat([h_base[neg_row], h_base[neg_col]], dim=-1)
    
    lp_pos_b = model.link_predictor(pos_base_feat).squeeze()
    lp_neg_b = model.link_predictor(neg_base_feat).squeeze()
    
    y_pred_b = torch.cat([lp_pos_b, lp_neg_b])
    global_base_acc = ((y_pred_b > 0.5).float() == y_true).float().mean().item()
    
    return density, active_edges, total_edges, sampler_impact_acc, global_base_acc

# ==========================================
# 在训练 100 epoch 后的调用逻辑
# ==========================================
density, active, total, s_acc, g_acc = evaluate_density_and_link_impact()


print(f"【Amazon】")
print("-" * 50)
print(f"1. 采样密度分析 (Structural Sparsity):")
print(f"   - 激活边数 / 总边数: {active} / {total}")
print(f"   - 密度 (Density): {density:.2%}")
print("-" * 50)
print(f"2. 链路预测对比 (Link Prediction Impact):")
print(f"   - 全局基准 (无采样增强) Acc: {g_acc:.4f}")
print(f"   - 采样器 (经过增强表征) Acc: {s_acc:.4f}")
print(f"   - 性能提升 (Gain): {((s_acc - g_acc)/g_acc):.2%}")
print("-" * 50)



【Amazon】
--------------------------------------------------
1. 采样密度分析 (Structural Sparsity):
   - 激活边数 / 总边数: 132144 / 491722
   - 密度 (Density): 26.87%
--------------------------------------------------
2. 链路预测对比 (Link Prediction Impact):
   - 全局基准 (无采样增强) Acc: 0.5000
   - 采样器 (经过增强表征) Acc: 0.9002
   - 性能提升 (Gain): 80.04%
--------------------------------------------------
